# Image captioning on Flickr8k: zero-shot vs fine-tuned BLIP (Kaggle, runs unattended)

Right panel: **Accelerator: GPU T4 x2**, **Internet: on**. **Add Input**: the Flickr8k dataset (`adityajn105/flickr8k`) and
your code dataset (`captioning-code`, the uploaded code zip). Then **Save Version > Save & Run All (Commit)**.

About 1-1.5 GPU hours: BLIP and GIT zero-shot, BLIP fine-tuned for 3 epochs, decoding chosen on validation, test metrics.
Output: `results.zip` (model version folder + log) in the notebook's Output tab.

In [ ]:
import glob, os, shutil, subprocess, zipfile
src = glob.glob('/kaggle/input/**/cap/train.py', recursive=True)
if not src:
    z = [p for p in glob.glob('/kaggle/input/**/*.zip', recursive=True)]
    print(z)
    zipfile.ZipFile(z[0]).extractall('/kaggle/working/code')
else:
    shutil.copytree(os.path.dirname(os.path.dirname(src[0])), '/kaggle/working/code', dirs_exist_ok=True)
os.chdir('/kaggle/working/code')
print(sorted(os.listdir('.')))
!pip -q install pycocoevalcap
DATA = os.path.dirname(sorted(glob.glob('/kaggle/input/**/captions.txt', recursive=True), key=len)[0])
print('data:', DATA, len(os.listdir(os.path.join(DATA, 'Images'))), 'images')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
cmd = (f'python -m cap.train --data "{DATA}" --out /kaggle/working/artifacts --epochs 2 --batch 16 --lr 2e-5 '
       f'--notes "BLIP fine-tuned 2 epochs, Kaggle T4" 2>&1 | grep --line-buffered -v -E "Warning|it/s\\]" | tee /kaggle/working/train_log.txt')
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
p.wait()

In [ ]:
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
shutil.copy('/kaggle/working/train_log.txt', '/kaggle/working/artifacts/train_log.txt')
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/artifacts')
print(round(os.path.getsize('/kaggle/working/results.zip') / 1e6), 'MB')
v = open('/kaggle/working/artifacts/LATEST').read().strip()
import json
m = json.load(open(f'/kaggle/working/artifacts/{v}/metrics.json'))
for k, r in m['results'].items():
    print(k, r['test'])